# Summarize Each Filing

| Item | Description |
|---|---|
| Goal | One summary per 10-K filing, written by the language model from the filing's Item 1 segments with the map-collapse-reduce of T1. Every segment is summarized (map). When the segment summaries exceed the aggregation budget they are condensed in groups (collapse). One call writes the filing's summary (reduce). |
| Input | `settings.json` and `filing_inputs/<set>/shard_<k>.parquet` from T3.3.1 in the T3.3 Interim folder. One run of this notebook does one shard of one input set for one prompt family; `SET`, `FAMILY`, and `SHARD` are set in Step 2. |
| Output | In the T3.3 Interim folder, `filing_summaries/<set>/<family>/<prompt version>/`: `shard_<k>.jsonl` (one line per filing), `shard_<k>.calls.jsonl` (every model call), and `shard_<k>.json` (the run record). |
| Model | `Qwen/Qwen3.5-4B` in bfloat16 on one GPU; `do_sample=True`, `temperature=0.1`, reasoning traces off; a seed per shard. Prompts of one stage are batched by length so similar-sized prompts share a forward pass. A filing summary over 512 encoder tokens is re-asked shorter, up to twice. |
| Run | Ran in October 2026 as two SLURM arrays of eight tasks on NVIDIA A40 cards (`J - Jobs/run_T3.3.2_summarize_filings.sbatch`): the three-year set, then the remaining filings, 18,779 filings in all. Both used the `summary_lines_of_business` family at prompt version `item1-mapreduce-1.3`. The three-year set was also run at versions 1.1 and 1.2 while the prompt was developed. Not rerun, because sampling would write new texts; Step 10 reads the run records of every shard back. |


## Step 1. Import packages


In [ ]:
import hashlib
import json
import os
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from IPython.display import display


## Step 2. Define file locations, the shard, and the batch settings

Change `SET`, `FAMILY`, and `SHARD` to run another shard; the SLURM wrapper sets them from the environment.


In [ ]:
PROJECT_FOLDER = Path.cwd().resolve().parents[2]  # <pipeline>/<section>/<subfolder>/ sits three levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("T - Three Step Text Analysis Pipeline/T3 - Topic Relevance and Corporate Disclosure/T3.3 - Firm Side - Summaries of the 10-K Item 1 Filings")
NOTEBOOK_NAME = "T3.3.2 - Production - Summarize Each Filing"  # the notebook filename without .ipynb
RAW_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Raw"
INTERIM_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Interim"
PROCESSED_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Processed"
SETTINGS_FILE = INTERIM_FOLDER / "settings.json"
INPUTS_FOLDER = INTERIM_FOLDER / "filing_inputs"
SUMMARIES_FOLDER = INTERIM_FOLDER / "filing_summaries"
ENCODER = "BAAI/bge-large-en-v1.5"  # its tokenizer counts the tokens of each summary against the 512 limit

SET = os.environ.get("SET", "three_years")  # "three_years" or "remaining"
FAMILY = os.environ.get("FAMILY", "summary_lines_of_business")
SHARD = int(os.environ.get("SLURM_ARRAY_TASK_ID", os.environ.get("SHARD", 0)))
LIMIT = None  # smoke test: only the first N filings of the shard
MAX_BATCH_TOKENS = 48_000  # total padded prompt tokens per forward pass; about 23 GB on an A40 for 55,000
MAX_BATCH_PROMPTS = 16  # prompts per forward pass at most
MAX_COLLAPSE_LEVELS = 8  # runaway guard on collapse rounds, as in T1
SEED = 0  # plus the shard number; sampling at temperature 0.1 is near-greedy
SHORTER_TARGET_WORDS = ["100 to 150", "80 to 100"]  # re-asks for a final summary that exceeds the encoder limit
PROGRESS_EVERY = 10  # batches between progress lines
FIGURE_FOLDER = PROJECT_FOLDER / "R - Results/figures" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
TABLE_FOLDER = PROJECT_FOLDER / "R - Results/tables" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
MODEL_CACHE = SCRATCH_FOLDER / "cache" / "huggingface"
ENV_FILE = PROJECT_FOLDER / "T - Three Step Text Analysis Pipeline/T3 - Topic Relevance and Corporate Disclosure" / ".env"
for folder in (INTERIM_FOLDER,):
    folder.mkdir(parents=True, exist_ok=True)


## Step 3. Point the Hugging Face cache at the scratch root


In [ ]:
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(MODEL_CACHE)
# A token is needed only for gated models. Qwen3.5-4B and bge-large-en-v1.5 are open; the lines stay so the cell also serves a gated model.
if ENV_FILE.exists():
    for line in ENV_FILE.read_text().splitlines():
        name, _, value = line.strip().partition("=")
        if name == "HF_TOKEN" and value:
            os.environ["HF_TOKEN"] = value.strip().strip('"').strip("'")


## Step 4. Read the settings and the shard

Finished filings are read back from an earlier session of the same shard and skipped, so an interrupted task can be resubmitted.


In [ ]:
settings = json.loads(SETTINGS_FILE.read_text(encoding="utf-8"))
assert FAMILY in settings["families"], f"unknown family {FAMILY}"
prompts = settings["prompts"]
shard = pd.read_parquet(INPUTS_FOLDER / SET / f"shard_{SHARD}.parquet")
if LIMIT is not None:
    shard = shard.loc[shard["accessionNo"].isin(shard["accessionNo"].drop_duplicates().head(LIMIT))].copy()
assert shard["n_model_tokens"].le(settings["segment_tokens"]).all()
output_folder = SUMMARIES_FOLDER / SET / FAMILY / settings["prompt_version"]
output_folder.mkdir(parents=True, exist_ok=True)
output_file = output_folder / f"shard_{SHARD}.jsonl"
calls_file = output_folder / f"shard_{SHARD}.calls.jsonl"
done = {json.loads(line)["accessionNo"] for line in output_file.read_text(encoding="utf-8").splitlines()} if output_file.exists() else set()
shard = shard.loc[~shard["accessionNo"].isin(done)].copy()
filings = shard.groupby("accessionNo", sort=False).first()[["gvkey", "conm", "fyear", "n_segments"]]
print(f"Set {SET}, family {FAMILY}, shard {SHARD}: {len(filings):,} filings to do ({len(done):,} already done), {len(shard):,} segments")


## Step 5. Load the language model and the encoder's tokenizer


In [ ]:
assert torch.cuda.is_available(), "run this notebook on a GPU node"
torch.manual_seed(SEED + SHARD)
tokenizer = AutoTokenizer.from_pretrained(settings["model"])
tokenizer.padding_side = "left"
pad_token_id = tokenizer.pad_token_id or tokenizer.eos_token_id
model = AutoModelForCausalLM.from_pretrained(settings["model"], dtype=torch.bfloat16, device_map="cuda").eval()
bge_tokenizer = AutoTokenizer.from_pretrained(ENCODER)
gpu_name = torch.cuda.get_device_name(0)
run_started = datetime.now(timezone.utc).isoformat(timespec="seconds")
calls_handle = calls_file.open("a", encoding="utf-8")
n_calls = 0
print(f"Model {settings['model']} on {gpu_name}, {next(model.parameters()).dtype}; prompt version {settings['prompt_version']}")


## Step 6. Define the batched generation

Prompts are sorted by length and grouped so a batch holds at most 16 prompts and 48,000 padded tokens. A completion that hits the cap is trimmed to its last sentence. Every completion goes to the call log with its stage, level, index, and an input hash, as T1 logs.


In [ ]:
def render(stage, user_prompt):
    """Chat template with the stage's system prompt and reasoning traces off."""
    messages = [{"role": "system", "content": prompts["system"][stage]}, {"role": "user", "content": user_prompt}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=settings["enable_thinking"])


def model_tokens(text):
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])


def trim_to_sentence(text):
    """Cut a capped completion back to its last complete sentence."""
    text = text.rstrip()
    if not text or text.endswith((".", "!", "?", '."', ".'", ".)")):
        return text
    cut = max(text.rfind(mark) for mark in ".!?")
    if cut <= 0:
        return text
    end = cut + 1
    while end < len(text) and text[end] in "\"')]":
        end += 1
    return text[:end].rstrip()


def clean(text):
    return re.sub(r"^<think>.*?</think>\s*", "", text.strip(), flags=re.DOTALL).strip()


def generate_batched(stage, items, max_new_tokens):
    """Generate completions for (key, user_prompt) items of one stage; returns {key: text}."""
    global n_calls
    rendered = {key: render(stage, user_prompt) for key, user_prompt in items}
    lengths = {key: model_tokens(text) for key, text in rendered.items()}
    order = sorted(rendered, key=lambda key: lengths[key])
    batches, current = [], []
    for key in order:
        longest = max([lengths[k] for k in current] + [lengths[key]])
        if current and (len(current) + 1 > MAX_BATCH_PROMPTS or longest * (len(current) + 1) > MAX_BATCH_TOKENS):
            batches.append(current)
            current = []
        current.append(key)
    if current:
        batches.append(current)
    results, user_prompts = {}, dict(items)
    for batch_number, batch in enumerate(batches, start=1):
        encoded = tokenizer([rendered[key] for key in batch], return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        started = time.time()
        with torch.no_grad():
            output_ids = model.generate(**encoded, max_new_tokens=max_new_tokens, pad_token_id=pad_token_id, **settings["generation"])
        generated = output_ids[:, encoded["input_ids"].shape[1]:].cpu()
        del output_ids, encoded
        torch.cuda.empty_cache()
        seconds = time.time() - started
        for row, key in enumerate(batch):
            ids = generated[row]
            ids = ids[ids != pad_token_id]
            text = clean(tokenizer.decode(ids, skip_special_tokens=True))
            hit_cap = int(len(ids)) >= max_new_tokens
            if hit_cap:
                text = trim_to_sentence(text)
            results[key] = text
            accessionNo, level, index = key
            calls_handle.write(json.dumps({"accessionNo": accessionNo, "family": FAMILY, "stage": stage, "level": level, "index": index,
                                           "prompt_version": settings["prompt_version"], "model": settings["model"],
                                           "input_sha256": hashlib.sha256(user_prompts[key].encode("utf-8")).hexdigest(), "input_tokens": lengths[key],
                                           "output_tokens": int(len(ids)), "hit_cap": hit_cap, "seconds_batch": round(seconds, 1), "batch_size": len(batch),
                                           "output": text, "at": datetime.now(timezone.utc).isoformat(timespec="seconds")}, ensure_ascii=False) + "\n")
            n_calls += 1
        calls_handle.flush()
        if batch_number % PROGRESS_EVERY == 0 or batch_number == len(batches):
            print(f"  {stage}: batch {batch_number}/{len(batches)}, {len(batch)} prompts, longest {max(lengths[k] for k in batch):,} tokens, {seconds:.0f}s", flush=True)
    return results


stage_started = time.time()


## Step 7. Map: one call per segment


In [ ]:
map_items = []
for segment in shard.itertuples(index=False):
    user_prompt = prompts["map_opening"].format(company=segment.conm, fyear=segment.fyear, segment=segment.segment_id, n_segments=segment.n_segments, text=segment.text) \
        + prompts["map_tasks"][FAMILY].format(target_words=settings["target_words"]["map"])
    map_items.append(((segment.accessionNo, 0, int(segment.segment_id)), user_prompt))
print(f"Map: {len(map_items):,} segment calls")
map_results = generate_batched("map", map_items, settings["max_new_tokens"]["map"]) if map_items else {}
summaries = {accessionNo: [map_results[(accessionNo, 0, segment_id)] for segment_id in range(1, int(n_segments) + 1)] for accessionNo, n_segments in filings["n_segments"].items()}
levels_reached = {accessionNo: 0 for accessionNo in summaries}


## Step 8. Collapse: condense segment summaries that exceed the aggregation budget

While a filing's segment summaries together exceed 6,144 model tokens, they are condensed in groups that fit the budget. The loop stops when every filing fits.


In [ ]:
join = "\n\n"
for level in range(1, MAX_COLLAPSE_LEVELS + 1):
    collapse_items, groups_of = [], {}
    for accessionNo, parts in summaries.items():
        if model_tokens(join.join(parts)) <= settings["aggregation_budget"]:
            continue
        groups, current, current_tokens = [], [], 0
        for part in parts:
            n_tokens = model_tokens(part) + (model_tokens(join) if current else 0)
            if current and current_tokens + n_tokens > settings["aggregation_budget"]:
                groups.append(current)
                current, current_tokens = [], 0
                n_tokens = model_tokens(part)
            current.append(part)
            current_tokens += n_tokens
        groups.append(current)
        groups_of[accessionNo] = len(groups)
        for index, group in enumerate(groups, start=1):
            filing = filings.loc[accessionNo]
            user_prompt = prompts["collapse_opening"].format(company=filing["conm"], fyear=filing["fyear"], text=join.join(group)) \
                + prompts["collapse_tasks"][FAMILY].format(target_words=settings["target_words"]["collapse"])
            collapse_items.append(((accessionNo, level, index), user_prompt))
    if not collapse_items:
        break
    print(f"Collapse level {level}: {len(groups_of):,} filings, {len(collapse_items):,} group calls")
    collapse_results = generate_batched("collapse", collapse_items, settings["max_new_tokens"]["collapse"])
    for accessionNo, n_groups in groups_of.items():
        summaries[accessionNo] = [collapse_results[(accessionNo, level, index)] for index in range(1, n_groups + 1)]
        levels_reached[accessionNo] = level
else:
    raise RuntimeError(f"segment summaries still over the aggregation budget after {MAX_COLLAPSE_LEVELS} collapse levels")


## Step 9. Reduce: one call per filing, the length rule, and the record

A summary over the encoder limit is re-asked with a shorter word target, up to twice. One line per filing goes to the shard file and a run record beside it.


In [ ]:
def reduce_prompt(accessionNo, target_words):
    filing = filings.loc[accessionNo]
    return prompts["reduce_opening"].format(company=filing["conm"], fyear=filing["fyear"], text=join.join(summaries[accessionNo])) \
        + prompts["reduce_tasks"][FAMILY].format(target_words=target_words)


reduce_items = [((accessionNo, 99, 1), reduce_prompt(accessionNo, settings["target_words"]["reduce"])) for accessionNo in summaries]
print(f"Reduce: {len(reduce_items):,} filing calls")
final = generate_batched("reduce", reduce_items, settings["max_new_tokens"]["reduce"]) if reduce_items else {}
rounds = {accessionNo: 1 for accessionNo in summaries}
for target_words in SHORTER_TARGET_WORDS:
    too_long = [accessionNo for accessionNo in summaries if len(bge_tokenizer(final[(accessionNo, 99, 1)])["input_ids"]) > settings["max_bge_tokens"]]
    if not too_long:
        break
    print(f"Re-asking {len(too_long):,} filings at {target_words} words")
    redo = generate_batched("reduce", [((accessionNo, 99, 1), reduce_prompt(accessionNo, target_words)) for accessionNo in too_long], settings["max_new_tokens"]["reduce"])
    for accessionNo in too_long:
        final[(accessionNo, 99, 1)] = redo[(accessionNo, 99, 1)]
        rounds[accessionNo] += 1

records_written = 0
with output_file.open("a", encoding="utf-8") as handle:
    for accessionNo, filing in filings.iterrows():
        text = final[(accessionNo, 99, 1)]
        n_bge_tokens = len(bge_tokenizer(text)["input_ids"])
        handle.write(json.dumps({"accessionNo": accessionNo, "gvkey": filing["gvkey"], "conm": filing["conm"], "fyear": int(filing["fyear"]), "family": FAMILY,
                                 "prompt_version": settings["prompt_version"], "summary": text, "n_words": len(text.split()), "n_bge_tokens": n_bge_tokens,
                                 "within_bge_limit": n_bge_tokens <= settings["max_bge_tokens"], "n_segments": int(filing["n_segments"]),
                                 "collapse_levels": levels_reached[accessionNo], "reduce_rounds": rounds[accessionNo], "run_started": run_started}, ensure_ascii=False) + "\n")
        records_written += 1
calls_handle.close()
seconds = time.time() - stage_started
(output_folder / f"shard_{SHARD}.json").write_text(json.dumps({
    "set": SET, "family": FAMILY, "shard_id": SHARD, "filings_done_this_run": records_written, "filings_skipped_as_done": len(done), "segments": int(len(shard)),
    "model_calls": n_calls, "prompt_version": settings["prompt_version"], "model": settings["model"], "gpu": gpu_name, "generation": settings["generation"],
    "enable_thinking": settings["enable_thinking"], "max_new_tokens": settings["max_new_tokens"], "seed": SEED + SHARD, "max_batch_tokens": MAX_BATCH_TOKENS,
    "max_batch_prompts": MAX_BATCH_PROMPTS, "torch": torch.__version__, "seconds": round(seconds, 1), "run_started": run_started,
    "finished": datetime.now(timezone.utc).isoformat(timespec="seconds")}, indent=2), encoding="utf-8")
print(f"Wrote {records_written:,} filing summaries ({n_calls:,} model calls) to {output_file.name} in {seconds / 60:.1f} minutes")


## Step 10. Read the run records of every shard back

Evidence of the October runs: one row per shard and run, with its filings, segments, model calls, card, and minutes. This cell runs without the model.


In [3]:
records = []
for record_file in sorted(SUMMARIES_FOLDER.glob("*/*/*/shard_*.json")):
    if re.fullmatch(r"shard_\d+\.json", record_file.name):
        record = json.loads(record_file.read_text(encoding="utf-8"))
        records.append({"set": record["set"], "family": record["family"], "prompt_version": record["prompt_version"], "shard": record["shard_id"], "filings": record["filings_done_this_run"] + record["filings_skipped_as_done"],
                        "segments": record["segments"], "model_calls": record["model_calls"], "gpu": record["gpu"], "minutes": round(record["seconds"] / 60, 1), "finished": record["finished"][:10]})
records = pd.DataFrame(records).sort_values(["prompt_version", "set", "shard"]).reset_index(drop=True)
totals = records.groupby(["prompt_version", "set"]).agg(shards=("shard", "size"), filings=("filings", "sum"), segments=("segments", "sum"), model_calls=("model_calls", "sum"), gpu_hours=("minutes", lambda m: round(m.sum() / 60, 1)))
display(totals)
display(records.loc[records["prompt_version"].eq("item1-mapreduce-1.3")].reset_index(drop=True))


shards  filings  segments  model_calls  \
prompt_version      set                                                   
item1-mapreduce-1.1 three_years       8     2072      6469         8542   
item1-mapreduce-1.2 three_years       8     2072      6469         8545   
item1-mapreduce-1.3 remaining         8    16707     51274        68079   
                    three_years       8     2072      6386         8439   

                                 gpu_hours  
prompt_version      set                     
item1-mapreduce-1.1 three_years        4.7  
item1-mapreduce-1.2 three_years        4.2  
item1-mapreduce-1.3 remaining         37.4  
                    three_years        4.6

,set,family,prompt_version,shard,filings,segments,model_calls,gpu,minutes,finished
0,remaining,summary_lines_of_business,item1-mapreduce-1.3,0,2089,5743,7840,NVIDIA A40,228.9,2026-10-26
1,remaining,summary_lines_of_business,item1-mapreduce-1.3,1,2088,5901,8000,NVIDIA A40,259.3,2026-10-26
2,remaining,summary_lines_of_business,item1-mapreduce-1.3,2,2089,5836,7929,NVIDIA A40,229.8,2026-10-26
3,remaining,summary_lines_of_business,item1-mapreduce-1.3,3,2088,5362,7456,NVIDIA A40,212.2,2026-10-26
4,remaining,summary_lines_of_business,item1-mapreduce-1.3,4,2088,7629,9739,NVIDIA A40,362.1,2026-10-26
5,remaining,summary_lines_of_business,item1-mapreduce-1.3,5,2089,6450,8548,NVIDIA A40,257.1,2026-10-26
6,remaining,summary_lines_of_business,item1-mapreduce-1.3,6,2088,7095,9199,NVIDIA A40,409.4,2026-10-26
7,remaining,summary_lines_of_business,item1-mapreduce-1.3,7,2088,7258,9368,NVIDIA A40,282.4,2026-10-26
8,three_years,summary_lines_of_business,item1-mapreduce-1.3,0,259,716,975,NVIDIA A40,31.3,2026-10-25
9,three_years,summary_lines_of_business,item1-mapreduce-1.3,1,259,788,1049,NVIDIA A40,33.8,2026-10-25
